# Conexión Mongo DB

In [8]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata'] 
coleccion = db['datos_scraping'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


# Conectar el Scraper a la BD y guardar
Se usa el script de la semana 3

In [23]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time #para capturar la hora y fecha de obtención

NOMBRE_GRUPO = "Energy" #Agegaremos el nombre de grupo dentro del diccionario y la hora en que se obtubo

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES (LA MEMORIA) ---
datos_finales = []
limite_paginas = 3

try:
    driver.get("https://datasource.kapsarc.org/explore/?orderBy=updated_at+DESC")
    
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Nivel de Profundidad {nivel_pagina + 1} ---")
        
        # 1. Espera al Contenedor Raíz (el que agrupa todos los resultados)
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "ods-catalog-card"))
        )

        # 2. Captura de Bloques de Primer Nivel (la "tarjeta" o "card" del item)
        bloques_primer_nivel = driver.find_elements(By.CLASS_NAME, "ods-catalog-card")
        
        
        for bloque in bloques_primer_nivel:
            # 3. Búsqueda Detallada dentro del bloque (Selectores específicos)
            # Extraemos el atributo 'title' del enlace para evitar texto truncado
            dato_identificador = bloque.find_element(By.TAG_NAME, "h3").find_element(By.TAG_NAME, "a").get_attribute("title")
            dato_valor = bloque.find_element(By.CLASS_NAME, "ods-catalog-card__title").text
            
            # Guardamos la estructura en nuestra memoria
            item_extraido = {
                "identificador": dato_identificador,
                "valor_sucio": dato_valor,
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": "Energy"
            }
            datos_finales.append(item_extraido)

        # --- NAVEGACIÓN: Lógica de Salto de Nodo (Paginación) ---
        try:
            # Buscamos el activador del siguiente nodo de datos
            disparador_siguiente = driver.find_element(By.CSS_SELECTOR, "aria-label='Next'")
            disparador_siguiente.click()
        except Exception:
            print("Fin del árbol de navegación o nodo no encontrado.")
            break 

    print(f"\nExtracción finalizada. Registros en memoria: {len(datos_finales)}")

except Exception as e:
    print(f"Fallo en la arquitectura de scraping: {e}")

finally:
    driver.quit()

#TENEMOS EL DICCIONARIO CON UN FORMATO JSON, OBTENIDO DEL SCRIPT. AHORA LO INCLUIMOS EN LA BASE DE DATOS
#Cambios en los datos de salida
from pymongo import MongoClient



# 1. Configuracion de conexion
try:
    # 'database' es el nombre del servicio en el docker-compose
    client = MongoClient('database', 27017)
    db = client['proyecto_bigdata']
    coleccion = db['datos_scraping']
    print("CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.")
except Exception as e:
    print("ERROR DE CONEXION:", e)

# 2. Procesamiento e insercion de datos
registros_exitosos = 0
registros_fallidos = 0

for dato in datos_finales:
    try:
        # Limpieza de simbolos y conversion a float
        # Se eliminan simbolos de moneda, comas y espacios en blanco
        valor_raw = dato.get('valor_sucio', '0')
        numeros = re.findall(r'\d+', valor_raw.replace(',', ''))
        
        dato['valor'] = float("".join(numeros)) if numeros else 0.0
        del dato['valor_sucio']
        
        # Insercion individual
        coleccion.insert_one(dato)
        registros_exitosos += 1
        
    except ValueError:
        print("ADVERTENCIA: No se pudo procesar el valor:", valor_sucio)
        registros_fallidos += 1
    except Exception as e:
        print("ERROR EN INSERCION:", e)
        registros_fallidos += 1

print("RESUMEN DE CARGA:")
print("Registros exitosos:", registros_exitosos)
print("Registros fallidos:", registros_fallidos)

--- Procesando Nivel de Profundidad 1 ---
Fallo en la arquitectura de scraping: Message: 

CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.
RESUMEN DE CARGA:
Registros exitosos: 0
Registros fallidos: 0


In [24]:
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from pymongo import MongoClient

# --- CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
# Añadimos un User-Agent para evitar bloqueos
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
datos_finales = []
limite_paginas = 2
# URL EXACTA DEL CATÁLOGO
URL_BUSQUEDA = "https://google.com/www.kapsarc.org"

try:
    print("Iniciando navegador...")
    driver.get(URL_BUSQUEDA)
    
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")
        
        # ESPERA ACTIVA: Aumentamos a 20 segundos por si el sitio está lento
        try:
            WebDriverWait(driver, 20).until(
                EC.visibility_of_element_located((By.CLASS_NAME, "ods-catalog-card"))
            )
        except Exception:
            print("Error: Los elementos no cargaron a tiempo. Revisa tu conexión.")
            break

        # Captura de tarjetas
        bloques = driver.find_elements(By.CLASS_NAME, "ods-catalog-card")
        
        for bloque in bloques:
            try:
                titulo = bloque.find_element(By.CLASS_NAME, "ods-catalog-card__title").text
                # Algunos no tienen metadatos visibles de inmediato
                try:
                    metadata = bloque.find_element(By.CLASS_NAME, "ods-catalog-card__metadata-item").text
                except:
                    metadata = "0"

                datos_finales.append({
                    "identificador": titulo,
                    "valor_sucio": metadata,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": "Energy"
                })
            except:
                continue

        # NAVEGACIÓN
        try:
            btn_next = driver.find_element(By.CSS_SELECTOR, "a[aria-label='Next']")
            driver.execute_script("arguments[0].click();", btn_next)
            time.sleep(5) # Pausa generosa para carga de JS
        except:
            break 

    print(f"Éxito: Se capturaron {len(datos_finales)} registros.")

except Exception as e:
    print(f"Error fatal: {e}")
finally:
    driver.quit()

# --- INSERCIÓN EN MONGODB ---
try:
    # Usa 'localhost' si ejecutas el script fuera de Docker, o 'database' si estás dentro
    client = MongoClient('localhost', 27017, serverSelectionTimeoutMS=5000)
    db = client['proyecto_bigdata']
    coleccion = db['datos_scraping']
    
    insertados = 0
    for d in datos_finales:
        # Extraer solo números de "1,200 records"
        nums = re.findall(r'\d+', d['valor_sucio'].replace(',', ''))
        d['valor'] = float("".join(nums)) if nums else 0.0
        del d['valor_sucio']
        
        coleccion.insert_one(d)
        insertados += 1
    
    print(f"MongoDB: {insertados} documentos guardados correctamente.")
except Exception as e:
    print(f"Error MongoDB: {e}")


Iniciando navegador...
--- Procesando Página 1 ---
Error: Los elementos no cargaron a tiempo. Revisa tu conexión.
Éxito: Se capturaron 0 registros.
MongoDB: 0 documentos guardados correctamente.


In [25]:
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from pymongo import MongoClient

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

datos_finales = []
limite_paginas = 3
# Nueva página recomendada: Frases célebres (ideal para práctica)
URL_PRACTICA = "http://toscrape.com" 

try:
    print(f"Conectando a la página de frases...")
    driver.get(URL_PRACTICA)
    
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Nivel de Profundidad {nivel_pagina + 1} ---")
        
        # 1. Espera al Contenedor (En esta página cada bloque es 'quote')
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "quote"))
        )

        # 2. Captura de Bloques
        bloques = driver.find_elements(By.CLASS_NAME, "quote")
        
        for bloque in bloques:
            # 3. Búsqueda Detallada (Texto de la frase y Autor)
            dato_identificador = bloque.find_element(By.CLASS_NAME, "text").text
            dato_valor = bloque.find_element(By.CLASS_NAME, "author").text
            
            item_extraido = {
                "identificador": dato_identificador[:50] + "...", # Truncamos para el log
                "valor_sucio": dato_valor,
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": "Frases_Energia" # Mantenemos el nombre de tu grupo
            }
            datos_finales.append(item_extraido)

        # --- NAVEGACIÓN: Paginación (Botón 'Next') ---
        try:
            # En esta página el selector es li.next a
            btn_next = driver.find_element(By.CSS_SELECTOR, "li.next a")
            btn_next.click()
            time.sleep(1) # Carga muy rápido
        except:
            print("Fin de la navegación o no hay más páginas.")
            break 

    print(f"\nExtracción finalizada. Registros en memoria: {len(datos_finales)}")

except Exception as e:
    print(f"Fallo en el scraping: {e}")
finally:
    driver.quit()

# --- PASO 2: INSERCIÓN EN MONGODB ---
try:
    # 'database' si usas Docker, 'localhost' si es local
    client = MongoClient('database', 27017, serverSelectionTimeoutMS=2000)
    db = client['proyecto_bigdata']
    coleccion = db['datos_scraping']
    
    registros_ok = 0
    for dato in datos_finales:
        try:
            # Aquí no hay números que limpiar, guardamos el autor como valor
            dato['valor'] = dato['valor_sucio']
            del dato['valor_sucio']
            
            coleccion.insert_one(dato)
            registros_ok += 1
        except:
            continue

    print(f"RESUMEN MONGODB: {registros_ok} frases guardadas exitosamente.")
except Exception as e:
    print(f"Error de conexión a MongoDB: {e}")

Conectando a la página de frases...
--- Procesando Nivel de Profundidad 1 ---
Fallo en el scraping: Message: 

RESUMEN MONGODB: 0 frases guardadas exitosamente.


In [26]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from pymongo import MongoClient

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)
datos_finales = []

# URL de búsqueda de Google sobre Energía (Directa para evitar bloqueos)
URL_GOOGLE = "https://google.com"

try:
    print(f"Conectando a Google para extraer noticias...")
    driver.get(URL_GOOGLE)
    
    # 1. Espera a que aparezcan los resultados de búsqueda (clase 'g')
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div.g"))
    )

    # 2. Capturamos los primeros 5 resultados
    bloques = driver.find_elements(By.CSS_SELECTOR, "div.g")[:5]
    
    for i, bloque in enumerate(bloques):
        try:
            # Extraemos el título del resultado
            titulo = bloque.find_element(By.TAG_NAME, "h3").text
            # Extraemos el enlace
            enlace = bloque.find_element(By.TAG_NAME, "a").get_attribute("href")
            
            if titulo:
                datos_finales.append({
                    "identificador": titulo,
                    "valor": enlace, # Guardamos el link como valor
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": "Noticias_Energia"
                })
                print(f"Dato {i+1} capturado: {titulo[:30]}...")
        except:
            continue

    print(f"\nExtracción finalizada. Registros listos: {len(datos_finales)}")

except Exception as e:
    print(f"Fallo crítico: {e}")
finally:
    driver.quit()

# --- PASO 2: INSERCIÓN EN MONGODB ---
if datos_finales:
    try:
        # 'database' si usas Docker, 'localhost' si es local
        client = MongoClient('database', 27017, serverSelectionTimeoutMS=2000)
        db = client['proyecto_bigdata']
        coleccion = db['datos_scraping']
        
        exitos = 0
        for dato in datos_finales:
            coleccion.insert_one(dato)
            exitos += 1

        print(f"RESUMEN MONGODB: {exitos} registros guardados exitosamente.")
    except Exception as e:
        print(f"Error de conexión a MongoDB: {e}")
else:
    print("No se obtuvieron datos para guardar.")


Conectando a Google para extraer noticias...
Fallo crítico: Message: 
Stacktrace:
#0 0x65446a570a6a <unknown>
#1 0x654469f7fab5 <unknown>
#2 0x654469fd2676 <unknown>
#3 0x654469fd28b1 <unknown>
#4 0x65446a01d614 <unknown>
#5 0x65446a01a7b6 <unknown>
#6 0x654469fc5cbf <unknown>
#7 0x654469fc6a81 <unknown>
#8 0x65446a536a64 <unknown>
#9 0x65446a539951 <unknown>
#10 0x65446a52321e <unknown>
#11 0x65446a53a51e <unknown>
#12 0x65446a509be0 <unknown>
#13 0x65446a55d9b8 <unknown>
#14 0x65446a55db88 <unknown>
#15 0x65446a56f4de <unknown>
#16 0x76f62cb6fac3 <unknown>

No se obtuvieron datos para guardar.


In [27]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from pymongo import MongoClient

# --- CONFIGURACIÓN MINIMALISTA ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
datos_finales = []

# URL ultra ligera de tecnología y energía
URL_LIGHT = "https://ycombinator.com"

try:
    print(f"Conectando a {URL_LIGHT}...")
    driver.get(URL_LIGHT)
    
    # Espera manual simple (evita el error de Timeout de WebDriverWait)
    time.sleep(5)

    # Capturamos los títulos de las noticias (clase 'titleline')
    bloques = driver.find_elements(By.CLASS_NAME, "titleline")[:5]
    
    for i, bloque in enumerate(bloques):
        try:
            titulo = bloque.text
            enlace = bloque.find_element(By.TAG_NAME, "a").get_attribute("href")
            
            datos_finales.append({
                "identificador": titulo,
                "valor": enlace,
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": "Energia_Tech"
            })
            print(f"Dato {i+1} extraído.")
        except:
            continue

except Exception as e:
    print(f"Error en navegador: {e}")
finally:
    driver.quit()

# --- INSERCIÓN FORZADA EN MONGODB ---
if len(datos_finales) > 0:
    try:
        # Intenta 'database' (Docker) y si falla usa 'localhost'
        try:
            client = MongoClient('database', 27017, serverSelectionTimeoutMS=2000)
            client.server_info() 
        except:
            client = MongoClient('localhost', 27017, serverSelectionTimeoutMS=2000)
            
        db = client['proyecto_bigdata']
        coleccion = db['datos_scraping']
        
        coleccion.insert_many(datos_finales)
        print(f"ÉXITO: {len(datos_finales)} registros guardados en MongoDB.")
    except Exception as e:
        print(f"Error MongoDB: {e}")
else:
    print("No se capturó nada. Revisa si tu equipo tiene salida a internet.")


Conectando a https://ycombinator.com...
No se capturó nada. Revisa si tu equipo tiene salida a internet.
